# Challenge 2 — Pipelines, comparación y selección del modelo (Wine)

Reproduce los notebooks 3 y 4 del ejemplo de Breast Cancer:

- construye los pipelines (preprocesamiento + clasificador);
- compara **árbol de decisión, regresión logística y KNN**;
- optimiza hiperparámetros con `GridSearchCV` y validación cruzada estratificada **solo sobre `train`**;
- guarda el campeón y los metadatos del entrenamiento.

> Los preprocesadores no se ajustan antes de la validación cruzada: viven **dentro** de cada pipeline para que se reajusten en cada fold y no haya *data leakage*. El conjunto de prueba no se abre en este notebook.

In [1]:
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree


In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_RAW = PROJECT_DIR / "data" / "raw"
DATA_PROCESSED = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"
for directory in (DATA_RAW, DATA_PROCESSED, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42


In [3]:
train_df = pd.read_csv(DATA_PROCESSED / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

feature_columns = contract["features"]
assert set(feature_columns) == set(train_df.columns) - {contract["target"]}
print(f"Contrato validado para {len(feature_columns)} variables.")

X_train = train_df[feature_columns]
y_train = train_df[contract["target"]]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

Contrato validado para 13 variables.


## Procesamiento: dos estrategias de preprocesamiento (definidas, no ajustadas)

- **Árbol:** solo imputación (los árboles no necesitan estandarizar).
- **Regresión logística y KNN:** imputación + estandarización (el EDA mostró escalas muy distintas).

Aunque no hay valores faltantes, el pipeline incluye imputación con la mediana para soportar datos futuros incompletos. Como las 13 variables son medidas químicas ya calculadas, no se inventan nuevas variables sin una hipótesis de dominio.

In [4]:
numeric_for_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

numeric_scaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

tree_preprocessor = ColumnTransformer([
    ("numeric", numeric_for_tree, feature_columns),
], verbose_feature_names_out=False)

scaled_preprocessor = ColumnTransformer([
    ("numeric", numeric_scaled, feature_columns),
], verbose_feature_names_out=False)

preprocessors = {
    "tree": tree_preprocessor,
    "scaled": scaled_preprocessor,
}
joblib.dump(preprocessors, ARTIFACTS_DIR / "preprocessors.joblib")
preprocessors

{'tree': ColumnTransformer(transformers=[('numeric',
                                  Pipeline(steps=[('imputer',
                                                   SimpleImputer(strategy='median'))]),
                                  ['alcohol', 'malic_acid', 'ash',
                                   'alcalinity_of_ash', 'magnesium',
                                   'total_phenols', 'flavanoids',
                                   'nonflavanoid_phenols', 'proanthocyanins',
                                   'color_intensity', 'hue',
                                   'od280/od315_of_diluted_wines', 'proline'])],
                   verbose_feature_names_out=False),
 'scaled': ColumnTransformer(transformers=[('numeric',
                                  Pipeline(steps=[('imputer',
                                                   SimpleImputer(strategy='median')),
                                                  ('scaler', StandardScaler())]),
                                  ['a

## Candidatos y espacios de búsqueda

Como el problema tiene **3 clases**, la métrica de selección es `roc_auc_ovr` (AUC uno-contra-resto, promedio macro). Se registran también `accuracy` y `f1_macro` para desempatar y para dar contexto.

In [5]:
candidates = {
    "decision_tree": (
        Pipeline([
            ("preprocessing", clone(preprocessors["tree"])),
            ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE)),
        ]),
        {
            "classifier__criterion": ["gini", "entropy"],
            "classifier__max_depth": [2, 3, 4, 5, 6, 8, None],
            "classifier__min_samples_split": [2, 5, 10, 20],
        },
    ),
    "logistic_regression": (
        Pipeline([
            ("preprocessing", clone(preprocessors["scaled"])),
            ("classifier", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
        ]),
        {"classifier__C": [0.01, 0.1, 1, 10, 100]},
    ),
    "knn": (
        Pipeline([
            ("preprocessing", clone(preprocessors["scaled"])),
            ("classifier", KNeighborsClassifier()),
        ]),
        {
            "classifier__n_neighbors": [3, 5, 7, 9, 11, 15],
            "classifier__weights": ["uniform", "distance"],
        },
    ),
}

SCORING = {"roc_auc_ovr": "roc_auc_ovr", "accuracy": "accuracy", "f1_macro": "f1_macro"}
SELECTION_METRIC = "roc_auc_ovr"

In [6]:
searches = {}
rows = []

for name, (pipeline, param_grid) in candidates.items():
    search = GridSearchCV(
        pipeline,
        param_grid,
        scoring=SCORING,
        refit=SELECTION_METRIC,
        cv=cv,
        n_jobs=-1,
        return_train_score=True,
    )
    search.fit(X_train, y_train)
    searches[name] = search
    best = search.best_index_
    rows.append({
        "model": name,
        "cv_roc_auc_ovr": search.cv_results_["mean_test_roc_auc_ovr"][best],
        "cv_accuracy": search.cv_results_["mean_test_accuracy"][best],
        "cv_f1_macro": search.cv_results_["mean_test_f1_macro"][best],
        "best_params": json.dumps(search.best_params_),
    })

comparison = (
    pd.DataFrame(rows)
    .sort_values(["cv_roc_auc_ovr", "cv_f1_macro"], ascending=False)
    .reset_index(drop=True)
)
comparison

,model,cv_roc_auc_ovr,cv_accuracy,cv_f1_macro,best_params
0,logistic_regression,1.000000,0.971921,0.972223,"{""classifier__C"": 10}"
1,knn,0.999673,0.957882,0.958661,"{""classifier__n_neighbors"": 15, ""classifier__w..."
2,decision_tree,0.935639,0.902217,0.905759,"{""classifier__criterion"": ""gini"", ""classifier_..."


In [7]:
champion_name = comparison.loc[0, "model"]
champion = searches[champion_name].best_estimator_

comparison.to_csv(REPORTS_DIR / "cv_model_comparison.csv", index=False)
joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

training_metadata = {
    "champion": champion_name,
    "selection_metric": SELECTION_METRIC,
    "cv_folds": 5,
    "cv_score": float(comparison.loc[0, "cv_roc_auc_ovr"]),
    "cv_accuracy": float(comparison.loc[0, "cv_accuracy"]),
    "cv_f1_macro": float(comparison.loc[0, "cv_f1_macro"]),
    "best_params": searches[champion_name].best_params_,
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(
    json.dumps(training_metadata, indent=2), encoding="utf-8"
)
print("Modelo campeón:", champion_name)
print("ROC AUC (ovr) CV:", round(training_metadata["cv_score"], 4))
print("Accuracy CV     :", round(training_metadata["cv_accuracy"], 4))
print("F1 macro CV     :", round(training_metadata["cv_f1_macro"], 4))
print("Mejores parámetros:", training_metadata["best_params"])

Modelo campeón: logistic_regression
ROC AUC (ovr) CV: 1.0
Accuracy CV     : 0.9719
F1 macro CV     : 0.9722
Mejores parámetros: {'classifier__C': 10}


### Comprobación de que el campeón es un pipeline completo

El artefacto debe contener preprocesamiento **y** clasificador.

In [8]:
print(champion)
assert isinstance(champion, Pipeline)
assert list(champion.named_steps) == ["preprocessing", "classifier"]

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['alcohol', 'malic_acid',
                                                   'ash', 'alcalinity_of_ash',
                                                   'magnesium', 'total_phenols',
                                                   'flavanoids',
                                                   'nonflavanoid_phenols',
                                                   'proanthocyanins',
                                                   'color_intensity', 'hue',
                                     

## Inspección opcional: gap entre entrenamiento y validación de cada candidato

In [9]:
gap_rows = []
for name, search in searches.items():
    i = search.best_index_
    gap_rows.append({
        "model": name,
        "train_roc_auc_ovr": search.cv_results_["mean_train_roc_auc_ovr"][i],
        "val_roc_auc_ovr": search.cv_results_["mean_test_roc_auc_ovr"][i],
    })
gap = pd.DataFrame(gap_rows).set_index("model")
gap["gap"] = gap["train_roc_auc_ovr"] - gap["val_roc_auc_ovr"]
gap.round(4)

,train_roc_auc_ovr,val_roc_auc_ovr,gap
model,,,
decision_tree,0.9999,0.9356,0.0643
logistic_regression,1.0000,1.0000,0.0000
knn,1.0000,0.9997,0.0003


In [10]:
if champion_name == "decision_tree":
    plt.figure(figsize=(18, 8))
    plot_tree(
        champion.named_steps["classifier"],
        feature_names=feature_columns,
        class_names=contract["target_names"],
        filled=True,
    )
    plt.show()
else:
    print(f"El campeón es {champion_name}; no hay árbol que dibujar.")

El campeón es logistic_regression; no hay árbol que dibujar.


## Lectura de los resultados

| Modelo | ROC AUC (ovr) CV | Accuracy CV | F1 macro CV | Mejores hiperparámetros |
|---|---:|---:|---:|---|
| **Regresión logística** (campeón) | 1.0000 | 0.972 | 0.972 | `C=10` |
| KNN | 0.9997 | 0.958 | 0.959 | `n_neighbors=15`, `weights="distance"` |
| Árbol de decisión | 0.9356 | 0.902 | 0.906 | `gini`, `max_depth=4`, `min_samples_split=2` |

- **La regresión logística es el campeón**: lidera las tres métricas de validación cruzada. Que gane un modelo lineal es coherente con el EDA, que mostró clases muy separables a partir de unas pocas variables (`flavanoids`, `proline`, `od280/od315`); no hizo falta una frontera de decisión compleja.
- **Con AUC saturado (≈1.0) la métrica de selección apenas distingue entre regresión logística y KNN**; por eso se incluyeron `accuracy` y `f1_macro` y se usó `f1_macro` como criterio de desempate. Conviene no sobre-interpretar la diferencia entre ambos con solo 142 ejemplos.
- **El árbol es claramente el peor** (AUC 0.936 frente a ≈1.0) y es el único con brecha entrenamiento–validación apreciable (0.9999 frente a 0.9356): se ajusta bien al entrenamiento pero generaliza peor, es decir, tiene mayor varianza con tan pocos datos.
- **No hubo fuga de información**: el escalado vive dentro del pipeline y se reajusta en cada fold; `test` no se abrió en este notebook.